In [16]:
import random

# ── PROBLEM DEFINITION ───────────────────────────────────────────
ZONES = [
    {"name": "City Center",     "pop": 10000, "risk": 1.5, "econ_val": 1.5},
    {"name": "North District",  "pop":  8000, "risk": 1.0, "econ_val": 1.0},
    {"name": "East District",   "pop": 12000, "risk": 1.1, "econ_val": 0.9},
    {"name": "South District",  "pop":  6000, "risk": 0.9, "econ_val": 0.8},
    {"name": "West District",   "pop":  9000, "risk": 1.0, "econ_val": 1.1},
    {"name": "University Area", "pop": 15000, "risk": 1.7, "econ_val": 1.2},
    {"name": "Industrial Zone", "pop":  7000, "risk": 0.8, "econ_val": 2.0}, 
    {"name": "Suburb North",    "pop":  5000, "risk": 0.6, "econ_val": 0.5},
]

R0               = 2.5
BASE_COST_UNIT   = 100
RESIDUAL_SPREAD  = 0.02  
NUM_ZONES        = len(ZONES)

MAX_INFECTIONS = sum(z["pop"] * ((1 - 1/(R0 * z["risk"])) + RESIDUAL_SPREAD) for z in ZONES)
MAX_COST       = sum(z["pop"] * BASE_COST_UNIT * z["econ_val"] for z in ZONES)

In [17]:
def calculate_fitness(position):
    total_infected = 0
    total_cost     = 0

    for i, zone in enumerate(ZONES):
        restriction = position[i]
        pop = zone["pop"]
        eff_R = (R0 * zone["risk"]) * (1 - restriction)

        attack_rate = (1 - (1 / eff_R)) if eff_R > 1 else 0

        infected = pop * (attack_rate + RESIDUAL_SPREAD)
        cost = restriction * pop * BASE_COST_UNIT * zone["econ_val"]
        
        total_infected += infected
        total_cost     += cost

    # positive value: 0 is perfect, 1.0 is worst case.
    return (0.5 * (total_infected / MAX_INFECTIONS) + 0.5 * (total_cost / MAX_COST))

In [18]:
# ── PSO COMPONENTS ───────────────────────────────────────────────
class Particle:
    def __init__(self):
        self.position = [random.uniform(0, 1) for _ in range(NUM_ZONES)]
        self.velocity = [0.0] * NUM_ZONES
        self.pbest_pos = self.position[:] #shallow copy
        self.fit = calculate_fitness(self.position)
        # Initialize with current fitness
        self.pbest_fit = self.fit

    def move(self, gbest_pos):
        w, c1, c2 = 0.5, 1.5, 1.5 
        for i in range(NUM_ZONES):
            r1, r2 = random.random(), random.random()
            self.velocity[i] = (w * self.velocity[i]) + \
                               (c1 * r1 * (self.pbest_pos[i] - self.position[i])) + \
                               (c2 * r2 * (gbest_pos[i] - self.position[i]))
            self.position[i] = max(0, min(1, self.position[i] + self.velocity[i])) #restriction between 0 (0%) and 1 (100%)
        
        self.fit = calculate_fitness(self.position)
        # MINIMIZATION
        if self.fit < self.pbest_fit:
            self.pbest_fit, self.pbest_pos = self.fit, self.position[:]

In [19]:
class Swarm:
    def __init__(self, size):
        self.particles = [Particle() for _ in range(size)]
        self.gbest_pos = self.particles[0].position[:]
        self.gbest_fit = self.particles[0].fit
        
        for p in self.particles:
            if p.fit < self.gbest_fit:
                self.gbest_fit = p.fit
                self.gbest_pos = p.position[:]
        
        self.no_improve_count = 0

    def optimize(self, iterations, stall_limit):
        print(f"Starting PSO Minimization (Swarm Size: {len(self.particles)})...\n")
        
        for t in range(iterations):
            found_improvement = False
            for p in self.particles:
                p.move(self.gbest_pos)
                
                # MINIMIZATION
                if p.fit < (self.gbest_fit - 1e-7):
                    self.gbest_fit = p.fit
                    self.gbest_pos = p.position[:]
                    found_improvement = True
            
            if found_improvement:
                self.no_improve_count = 0
            else:
                self.no_improve_count += 1
            
            if t % 20 == 0:
                print(f"Iteration {t:3d} | Best Fitness: {self.gbest_fit:.6f}")

            if self.no_improve_count >= stall_limit:
                print(f"\nConverged early at iteration {t} (No improvement for {stall_limit} steps).")
                break
        else:
            print(f"\nReached maximum iterations ({iterations}).")

In [ ]:
# ── RESULTS ──────────────────────────────────────────────────────
def decode_results(best_policy, best_score):
    print(f"\n{'ZONE':<18} | {'ORIGINAL POP':>12} | {'RESTR.':>8} | {'INFECTED':>10} | {'ECON COST':>10}")
    print("-" * 75)
    
    total_inf, total_pop, total_cost = 0, 0, 0
    
    for i, zone in enumerate(ZONES):
        r = best_policy[i]
        pop = zone["pop"]
        eff_R = (R0 * zone["risk"]) * (1 - r)
        attack_rate = (1 - (1/eff_R)) if eff_R > 1 else 0
        
        inf = pop * (attack_rate + RESIDUAL_SPREAD)
        cost = r * pop * BASE_COST_UNIT * zone["econ_val"]
        
        total_inf += inf
        total_pop += pop
        total_cost += cost
        
        print(f"{zone['name']:<18} | {pop:>12,} | {r:>7.1%} | {inf:>10.0f} | {cost:>10.0f}")

    print("-" * 75)
    print(f"{'TOTAL':<18} | {total_pop:>12,} | {'-':>8} | {int(total_inf):>10,} | {int(total_cost):>10,}")
    print(f"\nFinal Best Fitness (Targeting 0): {best_score:.6f}")
    print(f"Final Population Impact: {int(total_inf):,} infected out of {total_pop:,} total citizens.")

sim_swarm = Swarm(size=50)
sim_swarm.optimize(iterations=200, stall_limit=40)
decode_results(sim_swarm.gbest_pos, sim_swarm.gbest_fit)

Starting PSO Minimization (Swarm Size: 40)...

Iteration   0 | Best Fitness: 0.384923
Iteration  20 | Best Fitness: 0.331232
Iteration  40 | Best Fitness: 0.329543
Iteration  60 | Best Fitness: 0.329403
Iteration  80 | Best Fitness: 0.329397
Iteration 100 | Best Fitness: 0.329397
Iteration 120 | Best Fitness: 0.329397

Converged early at iteration 137 (No improvement for 50 steps).

ZONE               | ORIGINAL POP |   RESTR. |   INFECTED |  ECON COST
---------------------------------------------------------------------------
City Center        |       10,000 |   73.3% |        200 |    1100000
North District     |        8,000 |   60.0% |        160 |     480000
East District      |       12,000 |   63.6% |        240 |     687273
South District     |        6,000 |   55.6% |        120 |     266667
West District      |        9,000 |   60.0% |        180 |     594001
University Area    |       15,000 |   76.5% |        300 |    1376471
Industrial Zone    |        7,000 |    0.0% |  